# Physics-Informed Diffusion-PINN MRI Reconstruction
## Notebook 01 — Environment Setup & Kaggle Dataset Setup

This reproducible notebook verifies system hardware (GPU/CUDA), system software (Python, Git), GitHub repository access, project dependencies, Kaggle CLI authentication, dataset configuration, dataset download execution, and HDF5 data integrity inspection.

In [ ]:
# SECTION 1 — Project Information
PROJECT_NAME = "mri-diffusion-pinn"
REPO_DEFAULT_URL = "https://github.com/Siva05-code/mri-diffusion-pinn.git"

print("========================================")
print("PROJECT INFORMATION")
print("========================================")
print(f"Project Name       : {PROJECT_NAME}")
print(f"Repository URL     : {REPO_DEFAULT_URL}")
print("Framework          : PyTorch Physics-Informed Diffusion-PINN")
print("========================================")

In [ ]:
# SECTION 2 — System Information
import os
import platform

print("========================================")
print("SYSTEM INFORMATION")
print("========================================")
print(f"Operating System   : {platform.system()} {platform.release()}")
print(f"Platform           : {platform.platform()}")
print(f"Working Directory  : {os.getcwd()}")
print(f"Inside Kaggle      : {os.path.exists('/kaggle')}")
print("========================================")

In [ ]:
# SECTION 3 — Python Information
import sys

print("========================================")
print("PYTHON INFORMATION")
print("========================================")
print(f"Python Executable  : {sys.executable}")
print(f"Python Version     : {sys.version.split()[0]}")
print("========================================")

In [ ]:
# SECTION 4 — PyTorch & GPU Verification
import torch

cuda_ok = torch.cuda.is_available()
print("========================================")
print("PYTORCH & GPU VERIFICATION")
print("========================================")
print(f"PyTorch Version    : {torch.__version__}")
print(f"CUDA Available     : {cuda_ok}")
if cuda_ok:
    print(f"CUDA Version       : {torch.version.cuda}")
    print(f"GPU Name           : {torch.cuda.get_device_name(0)}")
else:
    print("GPU Status         : No GPU detected (CPU mode)")
print("========================================")

In [ ]:
# SECTION 5 — Kaggle CLI Availability & Authentication
import subprocess

print("========================================")
print("KAGGLE CLI & AUTHENTICATION CHECK")
print("========================================")

try:
    cli_res = subprocess.run(["kaggle", "--version"], capture_output=True, text=True, check=True)
    print(f"Kaggle CLI Version : {cli_res.stdout.strip()}")
    cli_ok = True
except Exception:
    print("Kaggle CLI         : NOT AVAILABLE")
    cli_ok = False

# Check credentials presence safely
has_env = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
has_json = os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json"))
if has_env or has_json:
    print("Kaggle Credentials : Detected (Hidden)")
    auth_ok = True
else:
    print("Kaggle Credentials : NOT CONFIGURED")
    auth_ok = False
print("========================================")

In [ ]:
# SECTION 6 — Load Configuration & Display Dataset Slug
import yaml

print("========================================")
print("DATASET CONFIGURATION")
print("========================================")

config_path = "configs/base.yaml"
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    dataset_slug = cfg.get("data", {}).get("kaggle_dataset", "")
    data_dir = cfg.get("data", {}).get("data_dir", "/kaggle/working/data/fastmri")
    print(f"Dataset Slug       : {dataset_slug}")
    print(f"Target Data Dir    : {data_dir}")
else:
    print(f"ERROR: Config file '{config_path}' not found.")
print("========================================")

In [ ]:
# SECTION 7 — Run Dataset Downloader
print("========================================")
print("DATASET DOWNLOAD EXECUTION")
print("========================================")

# Invoke python -m src.data.download_kaggle
dl_res = subprocess.run([sys.executable, "-m", "src.data.download_kaggle"], capture_output=True, text=True)
print(dl_res.stdout)
if dl_res.returncode != 0:
    print("Download Warnings/Stderr:", dl_res.stderr)
print("========================================")

In [ ]:
# SECTION 8 — Inspect Downloaded HDF5 File Structure
from pathlib import Path
import h5py

print("========================================")
print("HDF5 DATASET INSPECTION")
print("========================================")

target_dir = Path(data_dir) if 'data_dir' in locals() else Path("/kaggle/working/data/fastmri")
h5_files = sorted(list(target_dir.glob("*.h5")) + list(target_dir.glob("*.hdf5")))

if h5_files:
    sample_file = h5_files[0]
    print(f"Inspecting Sample HDF5 File: {sample_file.name}")
    print(f"File Path                 : {sample_file.resolve()}")
    print(f"File Size                 : {sample_file.stat().st_size / (1024*1024):.2f} MB")
    
    with h5py.File(sample_file, "r") as hf:
        print("
HDF5 Keys:", list(hf.keys()))
        for key in hf.keys():
            item = hf[key]
            if isinstance(item, h5py.Dataset):
                print(f"  Dataset '{key}' -> Shape: {item.shape}, Dtype: {item.dtype}")
        
        print("
HDF5 Header Attributes:")
        for attr_k, attr_v in hf.attrs.items():
            print(f"  Attr '{attr_k}': {attr_v}")
else:
    print("No .h5 or .hdf5 files found in data directory yet.")
print("========================================")

In [ ]:
# SECTION 9 — Final Environment & Data Setup Report
print("========================================")
print("MRI DIFFUSION-PINN ENVIRONMENT & DATA REPORT")
print("========================================")
print(f"Python              : OK ({sys.version.split()[0]})")
print(f"PyTorch             : OK ({torch.__version__})")
print(f"CUDA / GPU          : {'OK (' + torch.cuda.get_device_name(0) + ')' if cuda_ok else 'CPU Mode'}")
print(f"Kaggle CLI          : {'OK' if cli_ok else 'NOT AVAILABLE'}")
print(f"Kaggle Auth         : {'OK' if auth_ok else 'NOT CONFIGURED'}")
print(f"Dataset Slug        : {dataset_slug if 'dataset_slug' in locals() else 'None'}")
print(f"HDF5 Files Found    : {len(h5_files) if 'h5_files' in locals() else 0}")
print("----------------------------------------")
print("Next Step:")
print("  Run 02_kspace_inspection.ipynb for k-space viz & Inverse FFT")
print("========================================")